In [1]:
commands = [
    "sudo apt update",
    "sudo apt install -y python3-pip",
    "pip3 install Jetson.GPIO"
]

for command in commands:
    print(f"Executing: {command}")
    !{command}


SyntaxError: invalid syntax (2975582616.py, line 1)

In [ ]:
import time
import Jetson.GPIO as GPIO

# =========================
# GPIO 설정
# =========================
# BOARD 모드 = 젯슨 오린 나노 40핀 헤더의 "물리 핀 번호" 기준
GPIO.setmode(GPIO.BOARD)
GPIO.setwarnings(False)

# =========================
# 핀 번호 설정
# =========================
# 앞 모터용 L298N
FRONT_IN1 = 11   # 앞 왼쪽 모터 방향 1
FRONT_IN2 = 13   # 앞 왼쪽 모터 방향 2
FRONT_IN3 = 15   # 앞 오른쪽 모터 방향 1
FRONT_IN4 = 16   # 앞 오른쪽 모터 방향 2

# 뒤 모터용 L298N
REAR_IN1 = 29    # 뒤 왼쪽 모터 방향 1
REAR_IN2 = 31    # 뒤 왼쪽 모터 방향 2
REAR_IN3 = 33    # 뒤 오른쪽 모터 방향 1
REAR_IN4 = 35    # 뒤 오른쪽 모터 방향 2

ALL_PINS = [
    FRONT_IN1, FRONT_IN2, FRONT_IN3, FRONT_IN4,
    REAR_IN1, REAR_IN2, REAR_IN3, REAR_IN4
]

# =========================
# 모터 반전 설정
# =========================
# 모터를 연결했을 때 한쪽 바퀴가 반대로 돌면
# 아래 값을 False에서 True로 바꾸면 됩니다.
# 또는 모터 선 OUT1/OUT2를 서로 바꿔도 됩니다.

REVERSE_FRONT_LEFT = False
REVERSE_FRONT_RIGHT = False
REVERSE_REAR_LEFT = False
REVERSE_REAR_RIGHT = False


# =========================
# 초기화
# =========================
for pin in ALL_PINS:
    GPIO.setup(pin, GPIO.OUT)
    GPIO.output(pin, GPIO.LOW)


# =========================
# 기본 모터 제어 함수
# =========================
def set_motor(pin1, pin2, direction, reverse=False):
    """
    direction:
        1  = 정방향
        -1 = 역방향
        0  = 정지
    reverse:
        True이면 정방향/역방향을 반대로 적용
    """

    if reverse:
        direction *= -1

    if direction == 1:
        GPIO.output(pin1, GPIO.HIGH)
        GPIO.output(pin2, GPIO.LOW)

    elif direction == -1:
        GPIO.output(pin1, GPIO.LOW)
        GPIO.output(pin2, GPIO.HIGH)

    else:
        GPIO.output(pin1, GPIO.LOW)
        GPIO.output(pin2, GPIO.LOW)


def set_all_motors(front_left, front_right, rear_left, rear_right):
    """
    각 모터 방향을 한 번에 설정합니다.
    1  = 앞으로
    -1 = 뒤로
    0  = 정지
    """

    set_motor(FRONT_IN1, FRONT_IN2, front_left, REVERSE_FRONT_LEFT)
    set_motor(FRONT_IN3, FRONT_IN4, front_right, REVERSE_FRONT_RIGHT)
    set_motor(REAR_IN1, REAR_IN2, rear_left, REVERSE_REAR_LEFT)
    set_motor(REAR_IN3, REAR_IN4, rear_right, REVERSE_REAR_RIGHT)


# =========================
# 자동차 동작 함수
# =========================
def stop():
    set_all_motors(0, 0, 0, 0)
    print("정지")


def forward():
    set_all_motors(1, 1, 1, 1)
    print("전진")


def backward():
    set_all_motors(-1, -1, -1, -1)
    print("후진")


def turn_left():
    # 제자리 좌회전: 왼쪽 바퀴 후진, 오른쪽 바퀴 전진
    set_all_motors(-1, 1, -1, 1)
    print("좌회전")


def turn_right():
    # 제자리 우회전: 왼쪽 바퀴 전진, 오른쪽 바퀴 후진
    set_all_motors(1, -1, 1, -1)
    print("우회전")


def gentle_left():
    # 부드러운 좌회전: 왼쪽 바퀴 정지, 오른쪽 바퀴 전진
    set_all_motors(0, 1, 0, 1)
    print("부드러운 좌회전")


def gentle_right():
    # 부드러운 우회전: 왼쪽 바퀴 전진, 오른쪽 바퀴 정지
    set_all_motors(1, 0, 1, 0)
    print("부드러운 우회전")


# =========================
# 키보드 조작 루프
# =========================
def main():
    print("===================================")
    print("젯슨 오린 나노 자동차 모터 테스트")
    print("===================================")
    print("w : 전진")
    print("s : 후진")
    print("a : 좌회전")
    print("d : 우회전")
    print("q : 부드러운 좌회전")
    print("e : 부드러운 우회전")
    print("x : 정지")
    print("exit : 종료")
    print("===================================")

    stop()

    try:
        while True:
            cmd = input("명령 입력 > ").strip().lower()

            if cmd == "w":
                forward()

            elif cmd == "s":
                backward()

            elif cmd == "a":
                turn_left()

            elif cmd == "d":
                turn_right()

            elif cmd == "q":
                gentle_left()

            elif cmd == "e":
                gentle_right()

            elif cmd == "x":
                stop()

            elif cmd == "exit":
                print("프로그램 종료")
                break

            else:
                print("알 수 없는 명령입니다. w/s/a/d/q/e/x/exit 중 입력하세요.")

    except KeyboardInterrupt:
        print("\n강제 종료")

    finally:
        stop()
        GPIO.cleanup()
        print("GPIO 정리 완료")


if __name__ == "__main__":
    main()